# PathSnap Step 2: Model Experiments (Gagana)

Trains and compares five classifiers with 5-fold grouped cross-validation on the 80 training trips, evaluates them on the 10 student-test trips, then runs Bayesian (Optuna TPE) tuning on the strongest family and compares tuned vs untuned.

**Attach data first:** Add data → *PathSnap Student Data* (shared by Juhi). Then Run All.

**Expected run time on Kaggle:** about 15 to 25 minutes. Outputs go to `/kaggle/working`: `model_comparison.csv`, `optuna_trials.csv`, `tuning_comparison.csv`, `artifacts/`.

**Time-limited choices (write these in the report):** CV folds fit on at most 200,000 sampled candidate rows; the Random Forest uses 50 trees; tuning uses 20 trials, 3 folds, a 15-minute cap.

In [1]:
%pip install -q optuna

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, glob, json, time
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

def find_root():
    env = os.environ.get("PATHSNAP_ROOT")
    if env and os.path.isdir(os.path.join(env, "train")):
        return env
    def search():
        hits = glob.glob("/kaggle/input/**/train", recursive=True) + glob.glob("/kaggle/working/raw/**/train", recursive=True)
        for h in hits:
            parent = os.path.dirname(h)
            if os.path.isdir(h) and os.path.isdir(os.path.join(parent, "student_test")):
                return parent
        return None
    root = search()
    if root is None:
        zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
        if zips:
            import zipfile
            zipfile.ZipFile(zips[0]).extractall("/kaggle/working/raw")
            root = search()
    if root is None:
        raise FileNotFoundError("Attach the PathSnap data: a folder with train/ and student_test/, or Student_folder.zip")
    return root

ROOT = find_root()
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "outputs"
os.makedirs(OUT, exist_ok=True)
print("ROOT:", ROOT)
print("OUT :", OUT)

ROOT: /kaggle/input/datasets/juhishahh/pathsnap-student-data/Student folder
OUT : /kaggle/working


In [3]:
R_CAND = 50.0    # metres: candidate radius and label radius
K_CAND = 40      # nearest arc midpoints examined per GPS point
FEATS = ["dist", "hd", "alen", "speed", "dt"]

def load_trip(folder, tid, with_route=False):
    d = os.path.join(folder, tid)
    track = np.loadtxt(os.path.join(d, tid + ".track"), ndmin=2)
    nodes = np.loadtxt(os.path.join(d, tid + ".nodes"), ndmin=2)
    arcs = np.loadtxt(os.path.join(d, tid + ".arcs"), dtype=int, ndmin=2)
    route = None
    if with_route:
        route = np.loadtxt(os.path.join(d, tid + ".route"), dtype=int, ndmin=1)
    return track, nodes, arcs, route

def project(lon, lat, lat0):
    # local equirectangular projection in metres (city scale)
    return lon * np.cos(np.radians(lat0)) * 111320.0, lat * 110574.0

def seg_dist(px, py, ax, ay, bx, by):
    abx, aby = bx - ax, by - ay
    t = np.clip(((px - ax) * abx + (py - ay) * aby) / (abx ** 2 + aby ** 2 + 1e-9), 0, 1)
    return np.hypot(px - (ax + t * abx), py - (ay + t * aby))

def connectivity(nodes, arcs):
    n = len(nodes)
    valid = (arcs[:, 0] >= 0) & (arcs[:, 0] < n) & (arcs[:, 1] >= 0) & (arcs[:, 1] < n)
    a = arcs[valid]
    g = coo_matrix((np.ones(len(a)), (a[:, 0], a[:, 1])), shape=(n, n))
    ncomp, _ = connected_components(g, directed=True, connection="weak")
    return ncomp

def candidates(track, nodes, arcs):
    """Candidate arcs per GPS observation. Does NOT use the reference route."""
    lat0 = track[:, 1].mean()
    nx, ny = project(nodes[:, 0], nodes[:, 1], lat0)
    s, e = arcs[:, 0], arcs[:, 1]
    ax, ay, bx, by = nx[s], ny[s], nx[e], ny[e]
    mx, my = (ax + bx) / 2, (ay + by) / 2
    alen = np.hypot(bx - ax, by - ay)
    abear = np.arctan2(bx - ax, by - ay)
    px, py = project(track[:, 0], track[:, 1], lat0)
    n = len(px)
    dx = np.r_[np.diff(px), 0.0]
    dy = np.r_[np.diff(py), 0.0]
    dt = np.r_[np.diff(track[:, 2]), 1.0]
    dt = np.where(dt <= 0, 1.0, dt)
    speed = np.hypot(dx, dy) / dt
    heading = np.arctan2(dx, dy)
    k = min(K_CAND, len(arcs))
    _, idx = cKDTree(np.column_stack([mx, my])).query(np.column_stack([px, py]), k=k)
    idx = np.asarray(idx).reshape(n, k)
    obs = np.repeat(np.arange(n), k)
    arc = idx.ravel()
    d = seg_dist(px[obs], py[obs], ax[arc], ay[arc], bx[arc], by[arc])
    hd = np.abs(np.angle(np.exp(1j * (heading[obs] - abear[arc]))))
    keep = d <= R_CAND
    return pd.DataFrame({
        "obs": obs[keep], "arc": arc[keep], "dist": d[keep], "hd": hd[keep],
        "alen": alen[arc[keep]], "speed": speed[obs[keep]], "dt": dt[obs[keep]],
    })

def reference_labels(track, nodes, arcs, route):
    """TRAINING AND SCORING ONLY. Nearest reference-route arc per GPS point (proxy label)."""
    lat0 = track[:, 1].mean()
    nx, ny = project(nodes[:, 0], nodes[:, 1], lat0)
    s, e = arcs[:, 0], arcs[:, 1]
    px, py = project(track[:, 0], track[:, 1], lat0)
    best = np.full(len(px), np.inf)
    lab = np.full(len(px), -1)
    for a in np.unique(route):
        d = seg_dist(px, py, nx[s[a]], ny[s[a]], nx[e[a]], ny[e[a]])
        m = d < best
        best[m] = d[m]
        lab[m] = a
    lab[best > R_CAND] = -1
    return lab

def trip_frames(folder, tid, with_route=False):
    track, nodes, arcs, route = load_trip(folder, tid, with_route)
    cand = candidates(track, nodes, arcs)
    cand.insert(0, "trip", tid)
    obs = pd.DataFrame({"trip": tid, "obs": np.arange(len(track)),
                        "lon": track[:, 0], "lat": track[:, 1], "t": track[:, 2]})
    if with_route:
        obs["label"] = reference_labels(track, nodes, arcs, route)
        cand = cand.merge(obs[["obs", "label"]], on="obs")
        cand["y"] = ((cand["arc"] == cand["label"]) & (cand["label"] >= 0)).astype(int)
    return cand, obs

In [4]:
def trips_in(part):
    base = os.path.join(ROOT, part)
    return sorted(d for d in os.listdir(base) if os.path.isdir(os.path.join(base, d)))

def build_feature_table(part):
    cs, os_ = [], []
    for tid in trips_in(part):
        c, o = trip_frames(os.path.join(ROOT, part), tid, with_route=True)
        cs.append(c)
        os_.append(o)
    return pd.concat(cs, ignore_index=True), pd.concat(os_, ignore_index=True)

import time
Ctr, Otr = build_feature_table("train")
Cte, Ote = build_feature_table("student_test")
print("train candidates", len(Ctr), "observations", len(Otr))
print("student_test candidates", len(Cte), "observations", len(Ote))

train candidates 1796686 observations 208417
student_test candidates 202711 observations 19160


In [5]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, log_loss
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
import joblib, json

SEED = 42
CV_ROWS = int(os.environ.get("PATHSNAP_CV_ROWS", 200000))
os.makedirs(os.path.join(OUT, "artifacts", "models"), exist_ok=True)
os.makedirs(os.path.join(OUT, "artifacts", "configs"), exist_ok=True)

def top1_rows(C):
    return C.loc[C.groupby(["trip", "obs"])["p"].idxmax()]

def metrics(C, O):
    sel = top1_rows(C)
    n_lab = int((O.label >= 0).sum())
    top1 = int((sel.y == 1).sum()) / max(n_lab, 1)
    present = int(C.groupby(["trip", "obs"])["y"].max().sum())
    miss = 1 - present / max(n_lab, 1)
    flag = C.index.isin(sel.index).astype(int)
    f1 = f1_score(C.y, flag, average="macro", zero_division=0)
    return {"top1": top1, "macro_f1": f1, "candidate_miss_rate": miss}

def test_eval(model):
    Ct = Cte.copy()
    Ct["p"] = model.predict_proba(Ct[FEATS])[:, 1]
    te = metrics(Ct, Ote)
    te["log_loss"] = log_loss(Ct.y, np.clip(Ct.p, 1e-6, 1 - 1e-6))
    return te

MODELS = {
    "logistic_regression": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=500)),
    "gaussian_nb": lambda: GaussianNB(),
    "decision_tree": lambda: DecisionTreeClassifier(max_depth=12, random_state=SEED),
    "random_forest": lambda: RandomForestClassifier(n_estimators=50, max_depth=12, n_jobs=-1, random_state=SEED),
    "hist_gradient_boosting": lambda: HistGradientBoostingClassifier(max_iter=200, random_state=SEED),
}

folds = list(GroupKFold(n_splits=5).split(Ctr, Ctr.y, groups=Ctr.trip))

def fit_sub(mk, idx):
    tr = Ctr.iloc[idx]
    tr = tr.sample(n=min(CV_ROWS, len(tr)), random_state=SEED)
    return mk().fit(tr[FEATS], tr.y)

rows, fitted = [], {}
for name, mk in MODELS.items():
    cvm = []
    for tr_idx, va_idx in folds:
        m = fit_sub(mk, tr_idx)
        Cv = Ctr.iloc[va_idx].copy()
        Cv["p"] = m.predict_proba(Cv[FEATS])[:, 1]
        cvm.append(metrics(Cv, Otr[Otr.trip.isin(Cv.trip.unique())]))
    t0 = time.time()
    final = mk().fit(Ctr[FEATS], Ctr.y)
    fit_s = time.time() - t0
    t0 = time.time()
    te = test_eval(final)
    inf_s = time.time() - t0
    fitted[name] = final
    joblib.dump(final, os.path.join(OUT, "artifacts", "models", name + ".joblib"))
    rows.append({
        "model": name,
        "cv_top1": np.mean([d["top1"] for d in cvm]),
        "cv_macro_f1": np.mean([d["macro_f1"] for d in cvm]),
        "test_top1": te["top1"], "test_macro_f1": te["macro_f1"], "test_log_loss": te["log_loss"],
        "candidate_miss_rate": te["candidate_miss_rate"],
        "train_seconds": round(fit_s, 2), "inference_seconds": round(inf_s, 3),
    })
    print(name, {k: round(v, 4) if isinstance(v, float) else v for k, v in rows[-1].items() if k != "model"}, flush=True)

results = pd.DataFrame(rows).sort_values("cv_top1", ascending=False)
results.to_csv(os.path.join(OUT, "model_comparison.csv"), index=False)
print(results.to_string(index=False))
print("Best by CV top-1 (test set not used for selection):", results.iloc[0]["model"])

logistic_regression {'cv_top1': np.float64(0.7381), 'cv_macro_f1': np.float64(0.8533), 'test_top1': 0.7545, 'test_macro_f1': 0.868, 'test_log_loss': 0.1621, 'candidate_miss_rate': 0.0145, 'train_seconds': 1.81, 'inference_seconds': 0.142}
gaussian_nb {'cv_top1': np.float64(0.7079), 'cv_macro_f1': np.float64(0.8365), 'test_top1': 0.7118, 'test_macro_f1': 0.8443, 'test_log_loss': 0.2966, 'candidate_miss_rate': 0.0145, 'train_seconds': 0.28, 'inference_seconds': 0.116}
decision_tree {'cv_top1': np.float64(0.6664), 'cv_macro_f1': np.float64(0.8133), 'test_top1': 0.7202, 'test_macro_f1': 0.849, 'test_log_loss': 0.1568, 'candidate_miss_rate': 0.0145, 'train_seconds': 15.29, 'inference_seconds': 0.101}
random_forest {'cv_top1': np.float64(0.7131), 'cv_macro_f1': np.float64(0.8394), 'test_top1': 0.7301, 'test_macro_f1': 0.8545, 'test_log_loss': 0.141, 'candidate_miss_rate': 0.0145, 'train_seconds': 85.81, 'inference_seconds': 0.387}
hist_gradient_boosting {'cv_top1': np.float64(0.7174), 'cv_ma

In [6]:
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
except ImportError:
    optuna = None
    print("optuna not installed: run the %pip cell at the top, then re-run this cell")

TUNE_N_TRIALS = 20
TUNE_TIMEOUT_S = 900
folds3 = list(GroupKFold(n_splits=3).split(Ctr, Ctr.y, groups=Ctr.trip))

def cv3_top1(make_model):
    scores = []
    for tr_idx, va_idx in folds3:
        tr = Ctr.iloc[tr_idx]
        tr = tr.sample(n=min(CV_ROWS, len(tr)), random_state=SEED)
        m = make_model().fit(tr[FEATS], tr.y)
        Cv = Ctr.iloc[va_idx].copy()
        Cv["p"] = m.predict_proba(Cv[FEATS])[:, 1]
        scores.append(metrics(Cv, Otr[Otr.trip.isin(Cv.trip.unique())])["top1"])
    return float(np.mean(scores))

def objective(trial):
    p = dict(
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        max_leaf_nodes=trial.suggest_int("max_leaf_nodes", 15, 127),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 20, 200),
        l2_regularization=trial.suggest_float("l2_regularization", 1e-6, 1.0, log=True),
        max_iter=trial.suggest_int("max_iter", 100, 400),
    )
    return cv3_top1(lambda: HistGradientBoostingClassifier(random_state=SEED, **p))

study = None
if optuna is not None:
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=TUNE_N_TRIALS, timeout=TUNE_TIMEOUT_S)
    study.trials_dataframe().to_csv(os.path.join(OUT, "optuna_trials.csv"), index=False)
    print("completed trials:", len(study.trials), "| best 3-fold CV top-1:", round(study.best_value, 4))
    print("best params:", study.best_params)

completed trials: 20 | best 3-fold CV top-1: 0.7225
best params: {'learning_rate': 0.01588965297162943, 'max_leaf_nodes': 29, 'min_samples_leaf': 147, 'l2_regularization': 0.569363745587342, 'max_iter': 225}


In [7]:
if study is None:
    print("Tuning skipped (optuna missing).")
else:
    untuned_cv = cv3_top1(lambda: HistGradientBoostingClassifier(max_iter=200, random_state=SEED))
    tuned_cv = study.best_value
    t0 = time.time()
    tuned = HistGradientBoostingClassifier(random_state=SEED, **study.best_params).fit(Ctr[FEATS], Ctr.y)
    tuned_fit_s = time.time() - t0
    t0 = time.time()
    te_tuned = test_eval(tuned)
    tuned_inf_s = time.time() - t0
    te_untuned = test_eval(fitted["hist_gradient_boosting"])
    comp = pd.DataFrame([
        {"variant": "untuned HGB", "cv3_top1": untuned_cv, "test_top1": te_untuned["top1"],
         "test_macro_f1": te_untuned["macro_f1"], "test_log_loss": te_untuned["log_loss"]},
        {"variant": "tuned HGB (Optuna TPE)", "cv3_top1": tuned_cv, "test_top1": te_tuned["top1"],
         "test_macro_f1": te_tuned["macro_f1"], "test_log_loss": te_tuned["log_loss"]},
    ])
    comp.to_csv(os.path.join(OUT, "tuning_comparison.csv"), index=False)
    print(comp.to_string(index=False))
    print("tuning compute: trials", len(study.trials), "| final fit seconds", round(tuned_fit_s, 2),
          "| inference seconds", round(tuned_inf_s, 3))
    final_name = "hgb_tuned" if tuned_cv >= untuned_cv else "hist_gradient_boosting"
    if final_name == "hgb_tuned":
        joblib.dump(tuned, os.path.join(OUT, "artifacts", "models", "hgb_tuned.joblib"))
    json.dump({"best_model": final_name, "features": FEATS,
               "selected_by": "3-fold CV top-1, tuned vs untuned on the same protocol",
               "best_params": study.best_params},
              open(os.path.join(OUT, "artifacts", "configs", "best.json"), "w"), indent=2)
    print("final model saved as:", final_name)

               variant  cv3_top1  test_top1  test_macro_f1  test_log_loss
           untuned HGB  0.720292   0.730988       0.854972       0.141169
tuned HGB (Optuna TPE)  0.722522   0.732765       0.855961       0.142608
tuning compute: trials 20 | final fit seconds 21.79 | inference seconds 1.605
final model saved as: hgb_tuned


## Notes for the report

- Models: five families (linear, probabilistic, tree, ensemble, boosting). For each one, explain why it was considered, its strengths and limitations, the configuration and the loss used.
- Selection uses cross-validation top-1 on training trips. The student-test columns are reported, not used for selection.
- Overfitting check: compare CV top-1 with student-test top-1 for each model.
- The labels are the proxy nearest-reference-arc labels from Juhi's notebook. Say so.

In [10]:

import os
import shutil
import zipfile

source = "/kaggle/working/artifacts"
zip_path = "/kaggle/working/Blood_Bank_Model_Experiments.zip"

# Create ZIP
shutil.make_archive(
    zip_path.replace(".zip", ""),
    "zip",
    root_dir="/kaggle/working",
    base_dir="artifacts"
)

# Verify ZIP
with zipfile.ZipFile(zip_path, "r") as z:
    print("ZIP CONTENTS:")
    for file in z.namelist():
        print(file)

print("\nZIP CREATED SUCCESSFULLY!")
print("File size:", round(os.path.getsize(zip_path) / (1024 * 1024), 2), "MB")


ZIP CONTENTS:
artifacts/
artifacts/configs/
artifacts/models/
artifacts/configs/best.json
artifacts/models/decision_tree.joblib
artifacts/models/gaussian_nb.joblib
artifacts/models/logistic_regression.joblib
artifacts/models/random_forest.joblib
artifacts/models/hgb_tuned.joblib
artifacts/models/hist_gradient_boosting.joblib

ZIP CREATED SUCCESSFULLY!
File size: 7.24 MB
